In [1]:
import cv2
import mediapipe as mp
import numpy as np
from ultralytics import YOLO

In [1]:
# Detector LSCh Estilo YOLO - Versión Mejorada
import cv2
import mediapipe as mp
from ultralytics import YOLO
import numpy as np
from collections import deque
import time

class LSChDetector:
    def __init__(self, model_path="runs/detect/train/weights/best.pt"):
        # Carga de los pesos del modelo YOLO
        self.model = YOLO(model_path)
        
        # Configuración optimizada de MediaPipe
        self.mp_hands = mp.solutions.hands
        self.hands = self.mp_hands.Hands(
            static_image_mode=False,
            max_num_hands=1,
            min_detection_confidence=0.7,  # Umbral incrementado para optimizar precisión
            min_tracking_confidence=0.5    # Umbral incrementado para optimizar seguimiento
        )
        
        # Memoria intermedia para estabilizar predicciones
        self.prediction_buffer = deque(maxlen=5)
        
        # Métricas de rendimiento
        self.frame_count = 0
        self.fps_start_time = time.time()
        self.current_fps = 0
        
        # Región de Interés (ROI) dinámica basada en detección de extremidad
        self.roi_buffer = deque(maxlen=3)
        
    def get_hand_roi(self, frame, hand_landmarks):
        """Extrae región de interés basada en landmarks de la mano"""
        h, w = frame.shape[:2]
        
        # Extracción de coordenadas espaciales de los puntos clave
        x_coords = [lm.x * w for lm in hand_landmarks.landmark]
        y_coords = [lm.y * h for lm in hand_landmarks.landmark]
        
        # Determinación de la caja delimitadora con margen adicional
        x_min, x_max = int(min(x_coords)), int(max(x_coords))
        y_min, y_max = int(min(y_coords)), int(max(y_coords))
        
        # Incorporación de un margen de seguridad del 20%
        padding_x = int((x_max - x_min) * 0.2)
        padding_y = int((y_max - y_min) * 0.2)
        
        x_min = max(0, x_min - padding_x)
        y_min = max(0, y_min - padding_y)
        x_max = min(w, x_max + padding_x)
        y_max = min(h, y_max + padding_y)
        
        return (x_min, y_min, x_max, y_max)
    
    def smooth_prediction(self, letter, confidence):
        """Suaviza predicciones usando un buffer"""
        self.prediction_buffer.append((letter, confidence))
        
        # Verificación de muestras suficientes en el buffer
        if len(self.prediction_buffer) >= 3:
            # Cálculo de frecuencia de ocurrencia por clase
            letter_counts = {}
            total_conf = 0
            
            for pred_letter, pred_conf in self.prediction_buffer:
                if pred_letter not in letter_counts:
                    letter_counts[pred_letter] = {'count': 0, 'total_conf': 0}
                letter_counts[pred_letter]['count'] += 1
                letter_counts[pred_letter]['total_conf'] += pred_conf
            
            # Identificación de la clase predominante
            best_letter = max(letter_counts.keys(), 
                            key=lambda x: letter_counts[x]['count'])
            avg_conf = letter_counts[best_letter]['total_conf'] / letter_counts[best_letter]['count']
            
            return best_letter, avg_conf
        
        return letter, confidence
    
    def calculate_fps(self):
        """Calcula FPS en tiempo real"""
        self.frame_count += 1
        if self.frame_count % 10 == 0:  # Frecuencia de actualización establecida en 10 fotogramas
            current_time = time.time()
            elapsed = current_time - self.fps_start_time
            self.current_fps = 10 / elapsed
            self.fps_start_time = current_time
    
    def draw_info_panel(self, frame, letter=None, confidence=None, hand_detected=False):
        """Dibuja panel de información"""
        h, w = frame.shape[:2]
        
        # Elementos gráficos de fondo
        cv2.rectangle(frame, (10, 10), (300, 100), (0, 0, 0), -1)
        cv2.rectangle(frame, (10, 10), (300, 100), (255, 255, 255), 2)
        
        # Despliegue de métricas y estado
        cv2.putText(frame, f"FPS: {self.current_fps:.1f}", 
                   (20, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
        
        status = "MANO DETECTADA" if hand_detected else "SIN MANO"
        color = (0, 255, 0) if hand_detected else (0, 0, 255)
        cv2.putText(frame, f"Estado: {status}", 
                   (20, 55), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
        
        if letter and confidence:
            cv2.putText(frame, f"Letra: {letter} ({confidence:.1%})", 
                       (20, 75), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    
    def process_frame(self, frame):
        """Procesa un frame completo"""
        # Inversión horizontal de la imagen (efecto espejo)
        frame = cv2.flip(frame, 1)
        
        # Detección de extremidades utilizando MediaPipe
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_results = self.hands.process(rgb_frame)
        hand_detected = mp_results.multi_hand_landmarks is not None
        
        current_letter = None
        current_confidence = None
        
        # Ejecución condicional supeditada a detección positiva
        if hand_detected:
            hand_landmarks = mp_results.multi_hand_landmarks[0]
            
            # Determinación de la Región de Interés
            roi = self.get_hand_roi(frame, hand_landmarks)
            x_min, y_min, x_max, y_max = roi
            
            # Extracción de la sub-matriz correspondiente a la extremidad
            hand_region = frame[y_min:y_max, x_min:x_max]
            
            if hand_region.size > 0:
                # Ejecución del modelo YOLO sobre la Región de Interés
                results = self.model.predict(
                    source=hand_region,
                    conf=0.3,  # Umbral de confianza permisivo para maximizar recuperaciones
                    verbose=False,
                    stream=False
                )
                
                # Análisis y selección de detecciones
                best_detection = None
                best_conf = 0
                
                for result in results:
                    if result.boxes is not None:
                        for box in result.boxes:
                            conf = float(box.conf[0])
                            if conf > best_conf:
                                best_conf = conf
                                cls = int(box.cls[0])
                                letter = self.model.names[cls]
                                
                                # Traslación de coordenadas locales a globales
                                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                                x1, y1, x2, y2 = int(x1 + x_min), int(y1 + y_min), int(x2 + x_min), int(y2 + y_min)
                                
                                best_detection = {
                                    'letter': letter,
                                    'confidence': conf,
                                    'bbox': (x1, y1, x2, y2)
                                }
                
                # Renderizado de la detección con mayor grado de confianza
                if best_detection and best_detection['confidence'] > 0.4:
                    # Aplicación de estabilización a la predicción
                    smooth_letter, smooth_conf = self.smooth_prediction(
                        best_detection['letter'], 
                        best_detection['confidence']
                    )
                    
                    current_letter = smooth_letter
                    current_confidence = smooth_conf
                    
                    # Renderizado gráfico de la caja delimitadora
                    x1, y1, x2, y2 = best_detection['bbox']
                    color = (0, 255, 0) if smooth_conf > 0.7 else (0, 255, 255)
                    
                    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 3)
                    
                    # Renderizado optimizado de la etiqueta
                    label = f"{smooth_letter} {smooth_conf:.1%}"
                    font = cv2.FONT_HERSHEY_SIMPLEX
                    font_scale = 1.0
                    thickness = 2
                    
                    (text_w, text_h), _ = cv2.getTextSize(label, font, font_scale, thickness)
                    
                    # Renderizado del fondo de la etiqueta
                    cv2.rectangle(frame, (x1, y1 - text_h - 10), 
                                 (x1 + text_w + 10, y1), color, -1)
                    
                    # Renderizado del texto de la etiqueta
                    cv2.putText(frame, label, (x1 + 5, y1 - 5), 
                               font, font_scale, (0, 0, 0), thickness)
        else:
            # Purgado de la memoria intermedia ante ausencia de detección
            self.prediction_buffer.clear()
        
        # Renderizado del panel informativo
        self.draw_info_panel(frame, current_letter, current_confidence, hand_detected)
        
        # Actualización de métricas de rendimiento
        self.calculate_fps()
        
        return frame
    
    def run(self):
        """Ejecuta el detector principal"""
        cap = cv2.VideoCapture(0)
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
        cap.set(cv2.CAP_PROP_FPS, 30)  # Restricción de la tasa de captura a 30 FPS
        
        print("=== Detector LSCh Mejorado ===")
        print("Presiona 'q' para salir")
        print("Presiona 'r' para reiniciar buffer")
        
        try:
            while True:
                ret, frame = cap.read()
                if not ret:
                    print("Error al capturar frame")
                    break
                
                # Ejecución del pipeline de procesamiento visual
                processed_frame = self.process_frame(frame)
                
                # Despliegue de la imagen resultante
                cv2.imshow('LSCh Detector - Mejorado', processed_frame)
                
                # Interfaz de controles del usuario
                key = cv2.waitKey(1) & 0xFF
                if key == ord('q'):
                    break
                elif key == ord('r'):
                    self.prediction_buffer.clear()
                    print("Buffer de predicciones reiniciado")
        
        except KeyboardInterrupt:
            print("\nDeteniendo detector...")
        
        finally:
            cap.release()
            cv2.destroyAllWindows()
            self.hands.close()

# Punto de entrada principal de ejecución
if __name__ == "__main__":
    detector = LSChDetector()
    detector.run()

=== Detector LSCh Mejorado ===
Presiona 'q' para salir
Presiona 'r' para reiniciar buffer
